# LÀM SẠCH VĂN BẢN TIẾNG VIỆT VÀ CHE THÔNG TIN CÁ NHÂN (PII)

In [1]:
import os
import re
import unicodedata
import pandas as pd
import numpy as np

In [2]:
ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
PROC_DIR = os.path.join(ROOT, 'data', 'processed')

In [3]:
df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_raw.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_raw.csv'))
df_test = pd.read_csv(os.path.join(PROC_DIR, 'test_raw.csv'))

In [4]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 11426 entries, 0 to 11425
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   sentence         11426 non-null  str  
 1   sentiment        11426 non-null  int64
 2   topic            11426 non-null  int64
 3   sentiment_label  11426 non-null  str  
 4   topic_label      11426 non-null  str  
dtypes: int64(2), str(3)
memory usage: 1.5 MB


In [5]:
df_train.head()

,sentence,sentiment,topic,sentiment_label,topic_label
0,slide giáo trình đầy đủ .,2,1,positive,training_program
1,"nhiệt tình giảng dạy , gần gũi với sinh viên .",2,0,positive,lecturer
2,đi học đầy đủ full điểm chuyên cần .,0,1,negative,training_program
3,chưa áp dụng công nghệ thông tin và các thiết ...,0,0,negative,lecturer
4,"thầy giảng bài hay , có nhiều bài tập ví dụ ng...",2,0,positive,lecturer


## 1. CHUẨN HOÁ UNICODE NFC VÀ CHỮ THƯỜNG

In [6]:
#Thử trên 1 câu mẫu có dấu tổ hợp và khoảng trắng thừa
tempt = 'Thầy dạy rất nhiệt tìnhhhh   nhưng phòng học hơi nóng   '

In [7]:
tempt = unicodedata.normalize('NFC', tempt)
tempt = tempt.lower().strip()
tempt

'thầy dạy rất nhiệt tìnhhhh   nhưng phòng học hơi nóng'

In [8]:
#Rút gọn ký tự lặp từ 3 lần trở lên về 2 lần để không mất từ vựng
re.sub(r'(.)\1{2,}', r'\1\1', tempt)

'thầy dạy rất nhiệt tìnhh  nhưng phòng học hơi nóng'

## 2. XỬ LÝ TEENCODE VÀ VIẾT TẮT HỌC ĐƯỜNG

In [9]:
#Từ điển viết tắt phổ biến của sinh viên Việt Nam
teencode_dict = {
    'k': 'không', 'ko': 'không', 'hok': 'không', 'hong': 'không', 'khong': 'không',
    'dc': 'được', 'đc': 'được', 'duoc': 'được',
    'gv': 'giảng viên', 'sv': 'sinh viên', 'gvien': 'giảng viên',
    'vs': 'với', 'cx': 'cũng', 'r': 'rồi', 'bt': 'bình thường',
    'mn': 'mọi người', 'hs': 'học sinh', 'mk': 'mình', 'mik': 'mình',
    'nc': 'nói chuyện', 'ok': 'ổn', 'oke': 'ổn', 'j': 'gì', 'z': 'vậy', 'v': 'vậy',
    'bh': 'bây giờ', 'trc': 'trước', 'tl': 'tài liệu', 'th': 'thực hành'
}

In [10]:
tempt = 'gv dạy ko hiểu j hết trơn, bài tập khó quá ko làm đc'

In [11]:
#Chỉ thay khi từ viết tắt đứng độc lập theo ranh giới từ
re.sub(r'\b\w+\b', lambda m: teencode_dict.get(m.group(0), m.group(0)), tempt)

'giảng viên dạy không hiểu gì hết trơn, bài tập khó quá không làm được'

## 3. CHE THÔNG TIN CÁ NHÂN (PII SCRUBBING)

In [12]:
#Thử che MSSV, số điện thoại, email và tên giảng viên trên câu mẫu
tempt = 'Thầy Nguyễn Văn An dạy hay, em mssv 21520123 sđt 0912345678 mail sv@gm.uit.edu.vn'

In [13]:
#1. Che email
tempt = re.sub(r'[\w.+-]+@[\w-]+(\.[\w-]+)+', '[EMAIL]', tempt)
#2. Che số điện thoại VN
tempt = re.sub(r'(?<!\d)(?:\+?84|0)(?:[\s.-]?\d){8,10}(?!\d)', '[PHONE]', tempt)
#3. Che mã số sinh viên 7-10 chữ số
tempt = re.sub(r'(?i)\b(?:mssv|msv|mã\s*sv|ma\s*sv)\s*[:\-]?\s*\w{6,12}\b', '[STUDENT_ID]', tempt)
tempt = re.sub(r'(?<!\d)\d{7,10}(?!\d)', '[STUDENT_ID]', tempt)
tempt

'Thầy Nguyễn Văn An dạy hay, em [STUDENT_ID] sđt [PHONE] mail [EMAIL]'

In [14]:
#4. Che tên riêng sau chữ thầy/cô/giảng viên nhưng giữ lại danh xưng để câu còn nghĩa
tempt2 = 'Thầy Nguyễn Văn An và cô Trần Thị Bình dạy rất dễ hiểu'
re.sub(r'\b([Tt]hầy|[Cc]ô|GV|TS\.?|ThS\.?)\s+(?:[A-ZÀ-Ỵ][a-zà-ỹ]+)(?:\s+[A-ZÀ-Ỵ][a-zà-ỹ]+){0,2}', r'\1 [GV]', tempt2)

'Thầy [GV] và cô [GV] dạy rất dễ hiểu'

## 4. ÁP DỤNG LÀM SẠCH CHO CẢ 3 TẬP TRAIN / DEV / TEST

In [15]:
#Gom các bước vừa thử thành hàm clean_text để apply cho cả cột
def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text)
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'\b([Tt]hầy|[Cc]ô|GV|TS\.?|ThS\.?)\s+(?:[A-ZÀ-Ỵ][a-zà-ỹ]+)(?:\s+[A-ZÀ-Ỵ][a-zà-ỹ]+){0,2}', r'\1 [gv]', text)
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'[\w.+-]+@[\w-]+(\.[\w-]+)+', ' ', text)
    text = re.sub(r'(?<!\d)(?:\+?84|0)(?:[\s.-]?\d){8,10}(?!\d)', ' ', text)
    text = re.sub(r'(?<!\d)\d{7,10}(?!\d)', ' ', text)
    text = text.lower()
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'\b\w+\b', lambda m: teencode_dict.get(m.group(0), m.group(0)), text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [16]:
for df in [df_train, df_dev, df_test]:
    df['sentence_clean'] = df['sentence'].apply(clean_text)

In [17]:
df_train[['sentence', 'sentence_clean']].head(10)

,sentence,sentence_clean
0,slide giáo trình đầy đủ .,slide giáo trình đầy đủ .
1,"nhiệt tình giảng dạy , gần gũi với sinh viên .","nhiệt tình giảng dạy , gần gũi với sinh viên ."
2,đi học đầy đủ full điểm chuyên cần .,đi học đầy đủ full điểm chuyên cần .
3,chưa áp dụng công nghệ thông tin và các thiết ...,chưa áp dụng công nghệ thông tin và các thiết ...
4,"thầy giảng bài hay , có nhiều bài tập ví dụ ng...","thầy giảng bài hay , có nhiều bài tập ví dụ ng..."
5,"giảng viên đảm bảo thời gian lên lớp , tích cự...","giảng viên đảm bảo thời gian lên lớp , tích cự..."
6,"em sẽ nợ môn này , nhưng em sẽ học lại ở các h...","em sẽ nợ môn này , nhưng em sẽ học lại ở các h..."
7,"thời lượng học quá dài , không đảm bảo tiếp th...","thời lượng học quá dài , không đảm bảo tiếp th..."
8,"nội dung môn học có phần thiếu trọng tâm , hầu...","nội dung môn học có phần thiếu trọng tâm , hầu..."
9,cần nói rõ hơn bằng cách trình bày lên bảng th...,cần nói rõ hơn bằng cách trình bày lên bảng th...


## 5. ĐÁNH DẤU CÂU NGẮN / KHÔNG CÓ Ý KIẾN VÀ TÁCH TỪ GHÉP CHO PHOBERT

In [18]:
#Danh sách câu vô nghĩa hay gặp khi sinh viên điền khảo sát bắt buộc
no_content_set = {'', 'k', 'ko', 'khong', 'không', 'không có', 'không có ý kiến', 'không có góp ý', 'none', 'n/a', '-', '.', 'x', 'ổn', 'ok'}

In [19]:
for df in [df_train, df_dev, df_test]:
    df['word_count'] = df['sentence_clean'].apply(lambda s: len(str(s).split()))
    df['char_count'] = df['sentence_clean'].apply(lambda s: len(str(s)))
    df['is_no_content'] = df['sentence_clean'].apply(lambda s: s.strip(' .,-!?') in no_content_set or len(s.strip(' .,-!?')) < 3)

In [20]:
df_train['is_no_content'].value_counts()

is_no_content
False    11419
True         7
Name: count, dtype: int64

In [21]:
#Ghép các từ ghép học đường phổ biến bằng dấu gạch dưới cho PhoBERT
compound_words = [
    'giảng viên', 'sinh viên', 'thầy cô', 'bài tập', 'kiến thức', 'thực hành', 'lý thuyết',
    'phòng học', 'máy chiếu', 'điều hòa', 'máy lạnh', 'phòng máy', 'cơ sở vật chất',
    'chương trình', 'giáo trình', 'tài liệu', 'nhiệt tình', 'dễ hiểu', 'khó hiểu',
    'điểm danh', 'cuối kỳ', 'giữa kỳ', 'học phần', 'môn học', 'thời gian', 'tiết học'
]
def segment_vn(text):
    s = str(text)
    for cw in compound_words:
        s = re.sub(r'\b' + cw + r'\b', cw.replace(' ', '_'), s)
    return s

In [22]:
tempt = 'giảng viên dạy rất nhiệt tình và dễ hiểu nhưng phòng máy hơi nóng'
segment_vn(tempt)

'giảng_viên dạy rất nhiệt_tình và dễ_hiểu nhưng phòng_máy hơi nóng'

In [23]:
for df in [df_train, df_dev, df_test]:
    df['sentence_seg'] = df['sentence_clean'].apply(segment_vn)

In [24]:
df_train[['sentence_clean', 'sentence_seg', 'word_count']].head()

,sentence_clean,sentence_seg,word_count
0,slide giáo trình đầy đủ .,slide giáo_trình đầy đủ .,6
1,"nhiệt tình giảng dạy , gần gũi với sinh viên .","nhiệt_tình giảng dạy , gần gũi với sinh_viên .",11
2,đi học đầy đủ full điểm chuyên cần .,đi học đầy đủ full điểm chuyên cần .,9
3,chưa áp dụng công nghệ thông tin và các thiết ...,chưa áp dụng công nghệ thông tin và các thiết ...,18
4,"thầy giảng bài hay , có nhiều bài tập ví dụ ng...","thầy giảng bài hay , có nhiều bài_tập ví dụ ng...",15


In [25]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 11426 entries, 0 to 11425
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   sentence         11426 non-null  str  
 1   sentiment        11426 non-null  int64
 2   topic            11426 non-null  int64
 3   sentiment_label  11426 non-null  str  
 4   topic_label      11426 non-null  str  
 5   sentence_clean   11426 non-null  str  
 6   word_count       11426 non-null  int64
 7   char_count       11426 non-null  int64
 8   is_no_content    11426 non-null  bool 
 9   sentence_seg     11426 non-null  str  
dtypes: bool(1), int64(4), str(5)
memory usage: 3.5 MB


In [26]:
#Lưu 3 file đã làm sạch để dùng cho EDA, Baseline và PhoBERT
df_train.to_csv(os.path.join(PROC_DIR, 'train_clean.csv'), index=False, encoding='utf-8-sig')
df_dev.to_csv(os.path.join(PROC_DIR, 'dev_clean.csv'), index=False, encoding='utf-8-sig')
df_test.to_csv(os.path.join(PROC_DIR, 'test_clean.csv'), index=False, encoding='utf-8-sig')

In [27]:
print('Đã lưu train_clean, dev_clean, test_clean thành công!')

Đã lưu train_clean, dev_clean, test_clean thành công!
